# Stage 8 — Separate calibration, threshold selection, and final evaluation

The P95 and P90 probabilities are calibrated independently. Early official-validation dates fit a sigmoid/Platt mapping; later validation dates choose an F1 operating threshold. The test period is evaluated only after the model family, calibrator, and threshold are fixed.

In [ ]:
from pathlib import Path
import json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score,roc_auc_score,brier_score_loss,log_loss,precision_score,recall_score,f1_score,balanced_accuracy_score,confusion_matrix,precision_recall_curve,roc_curve)
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore",category=FutureWarning)
sns.set_theme(style="whitegrid")
ROOT=Path.cwd().resolve()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"/"processed"; MODEL_DIR=ROOT/"models"
TABLE_DIR=ROOT/"reports"/"tables"/"08_model_evaluation"; TABLE_DIR.mkdir(parents=True,exist_ok=True)
FIGURE_DIR=ROOT/"reports"/"figures"/"08_model_evaluation"; FIGURE_DIR.mkdir(parents=True,exist_ok=True)
COMMON_FEATURES=["lag1_peak_kw","lag1_mean_kw","lag1_energy_kwh","lag1_coverage_ratio","lag1_was_complete","peak_mean_7d","peak_mean_30d","peak_max_7d","peak_max_30d","peak_std_7d","peak_std_30d","energy_mean_7d","energy_mean_30d","lag1_peak_over_p95","lag1_peak_minus_p95","peak_mean_7d_over_30d","lag1_peak_zscore_30d","day_of_week_sin","day_of_week_cos","is_weekend","month_sin","month_cos"]
EXPERIMENTS={
 "p95":{"path":DATA/"05_model_ready_m0_p95.parquet","role":"primary","features":COMMON_FEATURES+["p95_high_count_7d","p95_high_count_30d","p95_days_since_last_high"]},
 "p90":{"path":DATA/"05_model_ready_m0_p90.parquet","role":"sensitivity","features":COMMON_FEATURES+["p90_high_count_7d","p90_high_count_30d","p90_days_since_last_high"]},
}

def save_table(df,name,title=None):
    df.to_csv(TABLE_DIR/name,index=False)
    if title: print(title)
    display(df)

def logit_probability(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

def apply_calibrator(calibrator,p):
    return calibrator.predict_proba(logit_probability(p))[:,1]

def choose_f1_threshold(y,p):
    precision,recall,thresholds=precision_recall_curve(y,p)
    scores=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
    best=np.flatnonzero(np.isclose(scores,np.nanmax(scores)))[-1]
    return float(thresholds[best]),pd.DataFrame({"threshold":thresholds,"precision":precision[:-1],"recall":recall[:-1],"f1":scores})

def metric_record(target,subset,probability_type,y,p,threshold):
    pred=(np.asarray(p)>=threshold).astype(int); eps=1e-7; clipped=np.clip(p,eps,1-eps)
    return {"target":target.upper(),"subset":subset,"probability_type":probability_type,"rows":len(y),"events":int(np.sum(y)),"threshold":threshold,"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"brier":brier_score_loss(y,p),"log_loss":log_loss(y,clipped,labels=[0,1]),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0),"balanced_accuracy":balanced_accuracy_score(y,pred)}

selection=pd.read_csv(ROOT/"reports"/"tables"/"07_model_training"/"07_05_selected_models.csv").set_index("target")
selection

## 8.1 Freeze models and partition official validation chronologically

In [ ]:
datasets={k:pd.read_parquet(v["path"]) for k,v in EXPERIMENTS.items()}
for df in datasets.values(): df["measurement_date"]=pd.to_datetime(df["measurement_date"])
partition_rows=[]
for key,df in datasets.items():
    valid=df.loc[df.split_name.eq("validation")]
    dates=np.array(sorted(valid.measurement_date.unique())); cut=len(dates)//2
    cal_dates=dates[:cut]; threshold_dates=dates[cut:]
    for name,values in [("calibration",cal_dates),("threshold_selection",threshold_dates)]:
        x=valid.loc[valid.measurement_date.isin(values)]
        partition_rows.append({"target":key.upper(),"partition":name,"start_date":x.measurement_date.min(),"end_date":x.measurement_date.max(),"unique_dates":x.measurement_date.nunique(),"rows":len(x),"events":int(x.target.sum()),"event_rate":x.target.mean(),"both_classes":x.target.nunique()==2})
partitions=pd.DataFrame(partition_rows)
save_table(partitions,"08_01_validation_partitions.csv","Chronological validation partitions")
assert partitions.both_classes.all()
assert partitions.groupby("target").apply(lambda x:x.iloc[0].end_date<x.iloc[1].start_date,include_groups=False).all()

## 8.2 Fit target-specific Platt calibrators and lock thresholds

In [ ]:
metric_rows=[]; prediction_frames=[]; threshold_curves={}; run_rows=[]
for key,spec in EXPERIMENTS.items():
    df=datasets[key]; features=spec["features"]
    family=selection.loc[key.upper(),"selected_family"]
    model=joblib.load(MODEL_DIR/f"07_{key}_selected_model_m0.joblib")
    valid=df.loc[df.split_name.eq("validation")].sort_values(["measurement_date","household_id"]).copy()
    dates=np.array(sorted(valid.measurement_date.unique())); cut=len(dates)//2
    calibration=valid.loc[valid.measurement_date.isin(dates[:cut])].copy()
    threshold_set=valid.loc[valid.measurement_date.isin(dates[cut:])].copy()
    raw_cal=model.predict_proba(calibration[features])[:,1]
    calibrator=LogisticRegression(C=1e6,solver="lbfgs",max_iter=2000,random_state=42)
    calibrator.fit(logit_probability(raw_cal),calibration.target.astype(int))
    raw_threshold=model.predict_proba(threshold_set[features])[:,1]
    calibrated_threshold=apply_calibrator(calibrator,raw_threshold)
    threshold,curve=choose_f1_threshold(threshold_set.target.astype(int).to_numpy(),calibrated_threshold)
    curve.insert(0,"target",key.upper()); threshold_curves[key]=curve
    joblib.dump(calibrator,MODEL_DIR/f"08_{key}_platt_calibrator_m0.joblib")
    threshold_payload={"target":key.upper(),"experiment_role":spec["role"],"selected_family":family,"calibration":"Platt sigmoid on logit raw probability","decision_metric":"F1 on later validation dates","threshold":threshold,"calibration_dates":{"start":str(calibration.measurement_date.min().date()),"end":str(calibration.measurement_date.max().date())},"threshold_selection_dates":{"start":str(threshold_set.measurement_date.min().date()),"end":str(threshold_set.measurement_date.max().date())},"test_used_for_selection":False}
    (MODEL_DIR/f"08_{key}_decision_threshold_m0.json").write_text(json.dumps(threshold_payload,indent=2),encoding="utf-8")
    run_rows.append(threshold_payload)
    for subset_name,subset in [("validation_calibration",calibration),("validation_threshold",threshold_set)]:
        raw=model.predict_proba(subset[features])[:,1]; calibrated=apply_calibrator(calibrator,raw)
        y=subset.target.astype(int).to_numpy()
        metric_rows.append(metric_record(key,subset_name,"raw",y,raw,.5))
        metric_rows.append(metric_record(key,subset_name,"platt_calibrated",y,calibrated,threshold))
    # Only after family, calibrator and threshold are frozen is the test period touched.
    test=df.loc[df.split_name.eq("test")].sort_values(["measurement_date","household_id"]).copy()
    raw_test=model.predict_proba(test[features])[:,1]; calibrated_test=apply_calibrator(calibrator,raw_test); y_test=test.target.astype(int).to_numpy()
    metric_rows.append(metric_record(key,"test","raw",y_test,raw_test,.5))
    metric_rows.append(metric_record(key,"test","platt_calibrated",y_test,calibrated_test,threshold))
    pred=test[["row_id","household_id","measurement_date","target"]].copy()
    pred["target_definition"]=key.upper(); pred["selected_family"]=family; pred["raw_probability"]=raw_test; pred["calibrated_probability"]=calibrated_test; pred["decision_threshold"]=threshold; pred["prediction"]=(calibrated_test>=threshold).astype("int8")
    prediction_frames.append(pred)

threshold_curve=pd.concat(threshold_curves.values(),ignore_index=True)
save_table(threshold_curve,"08_02_validation_threshold_curves.csv","Threshold-selection curves")
threshold_summary=pd.DataFrame(run_rows)
save_table(threshold_summary,"08_03_locked_thresholds.csv","Locked target-specific thresholds")
metrics=pd.DataFrame(metric_rows)
save_table(metrics,"08_04_evaluation_metrics.csv","Raw and calibrated evaluation metrics")
predictions=pd.concat(prediction_frames,ignore_index=True)
predictions.to_parquet(DATA/"06_test_predictions_m0.parquet",index=False)
save_table(predictions.groupby("target_definition",as_index=False).agg(rows=("row_id","size"),events=("target","sum"),predicted_events=("prediction","sum"),threshold=("decision_threshold","first")),"08_05_test_prediction_summary.csv","Test prediction summary")

## 8.3 Curves, calibration, and confusion matrices

In [ ]:
for key,spec in EXPERIMENTS.items():
    pred=predictions.loc[predictions.target_definition.eq(key.upper())]; y=pred.target.to_numpy()
    fig,axes=plt.subplots(1,2,figsize=(12,4.8))
    for col,label,color in [("raw_probability","Raw","#636363"),("calibrated_probability","Platt calibrated","#b2182b" if key=="p95" else "#2166ac")]:
        p=pred[col].to_numpy(); precision,recall,_=precision_recall_curve(y,p); fpr,tpr,_=roc_curve(y,p)
        axes[0].plot(recall,precision,label=f"{label} (AP={average_precision_score(y,p):.3f})",color=color)
        axes[1].plot(fpr,tpr,label=f"{label} (AUC={roc_auc_score(y,p):.3f})",color=color)
    axes[0].axhline(y.mean(),ls="--",color="grey",label=f"Prevalence={y.mean():.3f}"); axes[0].set(xlabel="Recall",ylabel="Precision",title=f"{key.upper()} test precision-recall")
    axes[1].plot([0,1],[0,1],"--",color="grey"); axes[1].set(xlabel="False-positive rate",ylabel="True-positive rate",title=f"{key.upper()} test ROC")
    for ax in axes: ax.legend()
    plt.tight_layout(); plt.savefig(FIGURE_DIR/f"08_{1 if key=='p95' else 2:02d}_{key}_test_pr_roc.png",dpi=180,bbox_inches="tight"); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,(key,spec) in zip(axes,EXPERIMENTS.items()):
    pred=predictions.loc[predictions.target_definition.eq(key.upper())]; y=pred.target.to_numpy()
    for col,label in [("raw_probability","Raw"),("calibrated_probability","Platt calibrated")]:
        observed,forecast=calibration_curve(y,pred[col],n_bins=10,strategy="quantile")
        ax.plot(forecast,observed,marker="o",label=label)
    ax.plot([0,1],[0,1],"--",color="grey"); ax.set(title=f"{key.upper()} test calibration",xlabel="Mean predicted probability",ylabel="Observed event rate"); ax.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR/"08_03_test_calibration.png",dpi=180,bbox_inches="tight"); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
for ax,key in zip(axes,["p95","p90"]):
    curve=threshold_curves[key]; selected=float(threshold_summary.loc[threshold_summary.target.eq(key.upper()),"threshold"].iloc[0])
    ax.plot(curve.threshold,curve.f1,label="F1"); ax.plot(curve.threshold,curve.precision,label="Precision",alpha=.8); ax.plot(curve.threshold,curve.recall,label="Recall",alpha=.8); ax.axvline(selected,color="black",ls="--",label=f"Selected={selected:.3f}")
    ax.set(title=f"{key.upper()} later-validation threshold",xlabel="Calibrated probability threshold",ylabel="Score",ylim=(0,1)); ax.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR/"08_04_validation_threshold_selection.png",dpi=180,bbox_inches="tight"); plt.show()

fig,axes=plt.subplots(1,2,figsize=(10,4.5))
for ax,key in zip(axes,["p95","p90"]):
    pred=predictions.loc[predictions.target_definition.eq(key.upper())]; cm=confusion_matrix(pred.target,pred.prediction,labels=[0,1])
    sns.heatmap(cm,annot=True,fmt=",",cmap="Blues",cbar=False,ax=ax); ax.set(title=f"{key.upper()} calibrated test decisions",xlabel="Predicted",ylabel="Observed",xticklabels=["Normal","Event"],yticklabels=["Normal","Event"])
plt.tight_layout(); plt.savefig(FIGURE_DIR/"08_05_test_confusion_matrices.png",dpi=180,bbox_inches="tight"); plt.show()

test_metrics=metrics.loc[(metrics.subset.eq("test")) & metrics.probability_type.eq("platt_calibrated")]
fig,ax=plt.subplots(figsize=(9,5)); plot=test_metrics.melt(id_vars="target",value_vars=["pr_auc","roc_auc","precision","recall","f1","balanced_accuracy"],var_name="metric",value_name="value")
sns.barplot(data=plot,x="metric",y="value",hue="target",palette={"P95":"#b2182b","P90":"#2166ac"},ax=ax); ax.set(title="Calibrated test performance: primary vs sensitivity",xlabel="",ylabel="Score",ylim=(0,1)); plt.xticks(rotation=20)
plt.tight_layout(); plt.savefig(FIGURE_DIR/"08_06_test_metric_comparison.png",dpi=180,bbox_inches="tight"); plt.show()

## 8.4 Final evaluation controls

In [ ]:
checks=[]
for key in EXPERIMENTS:
    p=partitions.loc[partitions.target.eq(key.upper())].reset_index(drop=True)
    t=threshold_summary.loc[threshold_summary.target.eq(key.upper())].iloc[0]
    pred=predictions.loc[predictions.target_definition.eq(key.upper())]
    checks.extend([
      (key,"calibration_before_threshold",p.loc[0,"end_date"]<p.loc[1,"start_date"]),
      (key,"both_classes_calibration",bool(p.loc[0,"both_classes"])),
      (key,"both_classes_threshold",bool(p.loc[1,"both_classes"])),
      (key,"test_not_used_for_selection",t["test_used_for_selection"] is False or t["test_used_for_selection"]==False),
      (key,"calibrator_saved",(MODEL_DIR/f"08_{key}_platt_calibrator_m0.joblib").exists()),
      (key,"threshold_saved",(MODEL_DIR/f"08_{key}_decision_threshold_m0.json").exists()),
      (key,"test_predictions_unique",pred.row_id.is_unique),
      (key,"probabilities_valid",pred.raw_probability.between(0,1).all() and pred.calibrated_probability.between(0,1).all()),
    ])
final_checks=pd.DataFrame(checks,columns=["target","check","passed"])
save_table(final_checks,"08_06_final_validation.csv","Stage 8 final validation")
assert final_checks.passed.all(),final_checks.loc[~final_checks.passed]
manifest={"stage":"08_model_evaluation","primary_target":"P95","sensitivity_target":"P90","calibration":"separate Platt sigmoid per target","threshold_selection":"maximum F1 on later validation dates","test_used_after_lock":True,"thresholds":threshold_summary.to_dict("records"),"test_metrics":test_metrics.to_dict("records"),"software":{"python":platform.python_version(),"pandas":pd.__version__,"sklearn":__import__('sklearn').__version__}}
(TABLE_DIR/"08_run_manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print("Stage 8 complete. P95 remains the primary scientific outcome; P90 is sensitivity evidence only.")

## Interpretation boundary

Calibration changes probability meaning but normally not ranking, so PR-AUC and ROC-AUC may remain nearly unchanged. The selected thresholds are operational choices optimized for F1 on this validation period; they are not universal clinical or grid-risk cutoffs. The short, late test calendar window limits claims about long-term temporal generalization.